Cztery komórki TODO uzupełnisz samodzielnie. Pełne funkcje oceny i poprawy polityki są do prześledzenia.

# Spotkanie 2 · Programowanie dynamiczne na planszy

**Pytanie.** Reguły ruchu są znane. Jak ocenić zadaną politykę i znaleźć lepszą dla całej planszy?

Po dwóch stanach robota przechodzimy do 16 pól. Po wejściu do celu lub dziury epizod się kończy. Najpierw rozpisz jeden ruch, potem wartość jednej akcji, a na końcu aktualizację wszystkich pól.

**Przed uruchomieniem.** Otwórz ten notatnik w Jupyterze z Pythonem 3. Plik `srodowiska.py` powinien znajdować się obok. Kod korzysta z biblioteki standardowej. Uruchamiaj komórki kolejno. Komórki z testami możesz powtórzyć po każdej zmianie.

Siatka ma cztery wiersze. Numery biegną od 0 do 15 wierszami. Cel to pole 15. Dziury to pola 5, 7, 11 i 12. Za wejście do celu dostajesz +1, za pozostałe ruchy 0.


In [ ]:
from math import isclose
from srodowiska import FrozenLake

sucha = FrozenLake(slippery=False)
sliska = FrozenLake(slippery=True)
LEWO, DOL, PRAWO, GORA = sucha.actions
KIERUNKI = dict(zip(sucha.actions, sucha.action_names))
SYMBOLE = dict(zip(sucha.actions, ("←", "↓", "→", "↑")))
GAMMA = 0.95

assert (sucha.n_states, sucha.start) == (16, 0)


## 1. Co mówi model

`env.outcomes(s, a)` zwraca wszystkie osiągalne skutki komendy `a` na polu `s`. Każdy wpis ma postać `(p, s_nast, r, koniec)`:

- `p` to szansa skutku po wybraniu komendy,
- `s_nast` to kolejne pole, `r` to nagroda za ten ruch,
- `koniec` informuje, czy epizod się skończył.

**Ćwiczenie.** Na kartce wypisz skutek suchej komendy „w lewo” z pola 0 i „w prawo” z pola 14. Potem uruchom kod i porównaj z modelem. Napisz funkcję, która odczytuje pojedynczy wynik na suchej planszy bez losowania.

**Pseudokod.** Rozpakuj jeden wiersz modelu. Upewnij się, że jego szansa wynosi 1. Zwróć trzy pozostałe pola.


In [ ]:
def rozpakuj_jedyny_wynik(env, s, a):
    # TODO 1. Odczytaj jeden wiersz env.outcomes(s, a).
    # Rozpakuj p, następny stan, nagrodę i znacznik końca.
    # Zwróć trzy ostatnie elementy w tej kolejności.
    raise NotImplementedError("Uzupełnij odczyt ruchu na suchej planszy.")


In [ ]:
assert rozpakuj_jedyny_wynik(sucha, 0, LEWO) == (0, 0, False)
assert rozpakuj_jedyny_wynik(sucha, 6, PRAWO) == (7, 0, True)
assert rozpakuj_jedyny_wynik(sucha, 14, PRAWO) == (15, 1, True)
assert set(sliska.outcomes(14, PRAWO)) == {
    (1 / 3, 10, 0, False), (1 / 3, 14, 0, False), (1 / 3, 15, 1, True)
}
assert {sp: p for p, sp, r, done in sliska.outcomes(0, LEWO)} == {0: 2 / 3, 4: 1 / 3}
print("OK: krawędź, dziura, cel, poślizg i połączenie dwóch gałęzi.")


## 2. Wartość jednej akcji

Dla bieżących oszacowań wartości $V_k$ jedna akcja ma wartość

$$q_k(s,a)=\sum_{s',r,d}p(s',r,d\mid s,a)
  \left[r+\gamma(1-d)V_k(s')\right].$$

$V_k(s')$ oznacza wartość kolejnego pola w tablicy z kroku $k$. $\gamma$ to dyskonto. Zmienna $d$ wynosi 1, gdy epizod kończy się po tym ruchu. W takiej gałęzi zostaje bieżąca nagroda.

**Ćwiczenie.** Przyjmij $V(13)=0{,}20$, $V(14)=0{,}60$, $V(10)=0{,}40$, $\gamma=0{,}95$. Oblicz cztery wartości akcji z pola 14 na sucho. Przy poślizgu dla „w prawo” odczytaj trzy skutki z poprzedniej komórki i zważ ich wkłady.

**Pseudokod.** Rozpocznij sumę od 0. Dla każdego skutku ustaw dalszy zwrot na 0 po końcu, w pozostałych przypadkach użyj $V_k(s')$. Dodaj iloczyn prawdopodobieństwa i całego wyrażenia w nawiasie.


In [ ]:
def wartosc_akcji(env, s, a, V, gamma):
    # TODO 2. Oblicz ważoną sumę wkładów wszystkich skutków.
    # W gałęzi kończącej epizod nie odczytuj V dla następnego stanu.
    raise NotImplementedError("Uzupełnij wartość jednej akcji.")


def wartosci_akcji(env, s, V, gamma):
    return [wartosc_akcji(env, s, a, V, gamma) for a in env.actions]


In [ ]:
V_przyklad = [0.0] * sucha.n_states
V_przyklad[13], V_przyklad[14], V_przyklad[10] = 0.20, 0.60, 0.40
V_przyklad[15] = 999.0  # Celowo błędny wpis. Po końcu nie wolno go użyć.

q_sucho = wartosci_akcji(sucha, 14, V_przyklad, GAMMA)
assert all(isclose(q, wynik) for q, wynik in zip(q_sucho, (0.19, 0.57, 1.0, 0.38)))
assert isclose(wartosc_akcji(sliska, 14, PRAWO, V_przyklad, GAMMA), 0.65)
print("OK: cztery suche wartości akcji i wartość ruchu w prawo na śliskiej planszy.")


## 3. Ocena polityki i poprawa decyzji

Polityka $\pi(a\mid s)$ to prawdopodobieństwo wyboru komendy $a$ w stanie $s$. W kroku oceny polityki ważymy wartości akcji tymi szansami.

$$V_{k+1}(s)=\sum_a \pi(a\mid s)\,q_k(s,a).$$

W polityce jednostajnej każda z czterech komend ma szansę $1/4$. Oblicz jej wartość po jednej aktualizacji na polu 14. Jak zmieni się wynik, jeżeli polityka wybiera tam zawsze „w prawo”? W obu przypadkach użyj suchej planszy i tych samych danych.

**Poprawa polityki.** Najpierw obliczamy $V^\pi$ dla ustalonej polityki. Następnie wybieramy akcje maksymalizujące $q$ wyznaczone z $V^\pi$. Ten wybór nazywamy zachłannym. Kolejne oceny i poprawy składają się na iterację polityki.


In [ ]:
def krok_oceny(env, pi, V, gamma):
    return [
        0.0 if env.terminal(s) else
        sum(p * q for p, q in zip(pi[s], wartosci_akcji(env, s, V, gamma)))
        for s in range(env.n_states)
    ]


def jednostajna(env):
    return [tuple([1 / len(env.actions)] * len(env.actions)) for _ in range(env.n_states)]


def deterministyczna(env, reguly):
    return [tuple(float(a == wybor) for a in env.actions) for wybor in reguly]


def ocen_polityke(env, pi, gamma, tol=1e-12, max_iter=10_000):
    V = [0.0] * env.n_states
    for _ in range(max_iter):
        V_nowe = krok_oceny(env, pi, V, gamma)
        delta = max(abs(a - b) for a, b in zip(V, V_nowe))
        if delta <= tol:
            return V_nowe
        V = V_nowe
    raise RuntimeError("Ocena polityki nie osiągnęła zadanej dokładności.")


def najlepsze_akcje(env, s, V, gamma, tol=1e-9):
    if env.terminal(s):
        return ()
    kandydaci = wartosci_akcji(env, s, V, gamma)
    maksimum = max(kandydaci)
    return tuple(a for a, q in zip(env.actions, kandydaci) if maksimum - q <= tol)


def iteruj_polityke(env, gamma, max_iter=100):
    # Zaczynamy od polityki deterministycznej: wszędzie lewo.
    reguly = [env.actions[0]] * env.n_states
    for numer in range(1, max_iter + 1):
        V = ocen_polityke(env, deterministyczna(env, reguly), gamma)
        poprawione = reguly.copy()
        for s in range(env.n_states):
            najlepsze = najlepsze_akcje(env, s, V, gamma)
            # Przy remisie zachowujemy poprzedni wybór, jeżeli jest jednym z najlepszych.
            if najlepsze and reguly[s] not in najlepsze:
                poprawione[s] = najlepsze[0]
        if poprawione == reguly:
            return V, reguly, numer
        reguly = poprawione
    raise RuntimeError("Polityka nie ustabilizowała się w zadanej liczbie ocen.")


pi_jednostajna = jednostajna(sucha)
pi_prawo = deterministyczna(sucha, [PRAWO] * sucha.n_states)
assert isclose(krok_oceny(sucha, pi_jednostajna, V_przyklad, GAMMA)[14], 0.535)
assert isclose(krok_oceny(sucha, pi_prawo, V_przyklad, GAMMA)[14], 1.0)
print("OK: z tych samych danych ocena jednostajna daje 0,535, a zawsze w prawo daje 1.")


### Sprawdzenie poprawy w jednym stanie

Zacznij od jednostajnej polityki na śliskiej planszy. Z jej oszacowanego $V^\pi$ oblicz wartości akcji z pola 14. Potem zmień politykę tylko na tym polu i oceń ją ponownie. Pierwsza liczba $q$ opisuje jeden wybrany ruch z dalszym stosowaniem starej polityki. Ponowna ocena uwzględnia, że odtąd przy **każdym** powrocie na pole 14 będziesz mieć nową regułę.


In [ ]:
pi_stara = jednostajna(sliska)
V_stare = ocen_polityke(sliska, pi_stara, GAMMA)
q_poprawa = wartosci_akcji(sliska, 14, V_stare, GAMMA)
najlepsze = najlepsze_akcje(sliska, 14, V_stare, GAMMA)
assert najlepsze == (DOL,)
pi_nowa = pi_stara.copy()
pi_nowa[14] = tuple(float(a == DOL) for a in sliska.actions)
V_nowe = ocen_polityke(sliska, pi_nowa, GAMMA)
assert V_nowe[14] > V_stare[14]
print("Wartości akcji starej polityki w polu 14:", [round(q, 6) for q in q_poprawa])
print(f"V starej polityki: {V_stare[14]:.6f}. V po zmianie tylko pola 14: {V_nowe[14]:.6f}.")


## 4. Iteracja wartości

W każdym polu porównujemy akcje i bierzemy maksimum:

$$V_{k+1}(s)=\max_a q_k(s,a).$$

Każdy przebieg korzysta z jednej starej tablicy $V_k$. Wszystkie wyniki zapisujemy do nowej tablicy. Pola terminalne mają wartość 0, ponieważ nagroda za wejście pojawia się w poprzednim przejściu. Po zbieżności odczytujemy najlepsze akcje, zachowując remisy.

**Pseudokod jednego przebiegu.** Dla każdego pola przeznacz osobny wpis w nowej tablicy. Jeśli pole jest terminalne, wpisz 0. W pozostałych przypadkach wyznacz wszystkie wartości akcji względem starej tablicy i zapisz maksimum.

**Pseudokod pętli.** Zacznij od zer. Po każdym przebiegu wyznacz $\max_s |V_{k+1}(s)-V_k(s)|$. Gdy ta różnica jest mniejsza od zadanej tolerancji lub jej równa, zwróć nową tablicę i liczbę przebiegów. Ustal limit przebiegów i zgłoś błąd, jeśli go osiągniesz bez zbieżności.


In [ ]:
def krok_wartosci(env, V, gamma):
    # TODO 3. Zbuduj nową tablicę, czytając wyłącznie z V.
    # W niekońcowych stanach weź maksimum wartości legalnych akcji.
    raise NotImplementedError("Uzupełnij synchroniczne maksimum Bellmana.")


def iteruj_wartosci(env, gamma, tol=1e-12, max_iter=10_000):
    # TODO 4. Wyzeruj V, wywołuj krok_wartosci i sprawdzaj zbieżność.
    # Zwróć parę (nowa_tablica, liczba_przebiegow).
    raise NotImplementedError("Uzupełnij pętlę iteracji wartości.")


In [ ]:
V_zero = [0.0] * sucha.n_states
V = krok_wartosci(sucha, V_zero, GAMMA)
assert V_zero == [0.0] * sucha.n_states, "Tablica wejściowa ma służyć tylko do odczytu."
assert V[14] == 1 and V[13] == 0 and V[15] == 0
assert isclose(krok_wartosci(sliska, V_zero, GAMMA)[14], 1 / 3)

for numer in range(2, 7):
    V = krok_wartosci(sucha, V, GAMMA)
    cel_testu = 0.0 if numer < 6 else GAMMA ** 5
    assert isclose(V[sucha.start], cel_testu, abs_tol=1e-12)

for env in (sucha, sliska):
    V_vi, przebiegi = iteruj_wartosci(env, GAMMA)
    V_pi, reguly, oceny = iteruj_polityke(env, GAMMA)
    assert all(isclose(a, b, abs_tol=1e-8) for a, b in zip(V_vi, V_pi))
    assert all(reguly[s] in najlepsze_akcje(env, s, V_vi, GAMMA)
               for s in range(env.n_states) if not env.terminal(s))

assert najlepsze_akcje(sucha, 0, V_zero, 0.0) == sucha.actions
assert najlepsze_akcje(sucha, 15, V_zero, GAMMA) == ()
print("OK: synchroniczność, terminale, propagacja od celu, zbieżność i remisy.")


## 5. Eksperyment. Co zmienia poślizg?

1. Przewidź, jak poślizg zmieni optymalną wartość pola startowego i najlepszą komendę na polu 14.
2. Porównaj plansze przy tym samym dyskoncie i tej samej nagrodzie. Zapisz obie wartości startu i najlepsze komendy z pola 14.
3. Wróć do `env.outcomes(14, a)`. Porównaj gałęzie wskazanych dwóch komend i ich wartości po zbieżności.
4. Na koniec porównaj wartość startu dla **tej samej jednostajnej polityki** na obu planszach. Wyjaśnij, jaki rozkład rzeczywistych kierunków daje symetryczny poślizg przy równomiernym wyborze komendy.

**Do oddania.** Uzupełniony kod, wyniki testów i dwa zdania wyjaśnienia wyniku eksperymentu.


In [ ]:
def pokaz_plansze(env, V, gamma):
    for wiersz in range(env.rows):
        wpisy = []
        poczatek = wiersz * env.cols
        for s in range(poczatek, poczatek + env.cols):
            decyzje = "".join(SYMBOLE[a] for a in najlepsze_akcje(env, s, V, gamma))
            wpisy.append(f"{s:2d} {V[s]:.4f} {decyzje or 'koniec':6s}")
        print(" | ".join(wpisy))


for nazwa, env in (("Sucha", sucha), ("Śliska", sliska)):
    V_jedn = ocen_polityke(env, jednostajna(env), GAMMA)
    V_star, przebiegi = iteruj_wartosci(env, GAMMA)
    print()
    print(f"{nazwa}. V polityki jednostajnej na starcie = {V_jedn[env.start]:.6f}")
    print(f"V* na starcie = {V_star[env.start]:.6f}. Przebiegi: {przebiegi}")
    print("Pole 14, najlepsze akcje:",
          ", ".join(KIERUNKI[a] for a in najlepsze_akcje(env, 14, V_star, GAMMA)))
    pokaz_plansze(env, V_star, GAMMA)
    for a in (PRAWO, DOL):
        q = wartosc_akcji(env, 14, a, V_star, GAMMA)
        print(f"Pole 14, {KIERUNKI[a]}: wartość {q:.6f}, gałęzie {env.outcomes(14, a)}")


## Przejście do następnej paczki

Tutaj w każdej aktualizacji możemy przeczytać **cały rozkład** skutków danego ruchu. W następnym ćwiczeniu dostaniemy pojedynczy zaobserwowany skutek lub całą zapisaną trajektorię. Wynik oceny polityki z tego notatnika będzie wtedy punktem odniesienia, z którym porównamy uczące się estymatory.

**Rozszerzenie programistyczne.** Wybierz własną politykę początkową do iteracji polityki. Porównaj końcową tablicę z wynikiem iteracji wartości. W przypadku remisu porównuj zbiór najlepszych akcji i wartości stanów.
